# 01. Data Inspection and Cleaning

In [1]:
import pandas as pd
from pathlib import Path

# File paths
raw_path = Path('../data/raw/cognivault_final_dataset.csv')
processed_dir = Path('../data/processed')
processed_path = processed_dir / 'cognivault_cleaned_dataset.csv'

# Load the raw dataset
df = pd.read_csv(raw_path)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Dataset shape: 2000 rows, 16 columns


,institution,degree,year,branch,living_arrangement,study_hours_outside_classes_per_day,academic_tasks_per_week,academic_workload,time_pressure,avg_sleep_hours,wakes_up_tired,physical_activity_days_per_week,difficulty_concentrating,academic_emotional_exhaustion,burnout_risk_score,burnout_risk_level
0,IIIT Pune,B.Tech,3,CSE,Hostel,3.5,8,4,4,5.2,5,2,3,4,57.1,High
1,IIIT Pune,B.Tech,3,CSE,Hostel,3.5,9,4,3,6.0,4,1,3,3,59.7,High
2,IIIT Pune,B.Tech,1,Cyber Security,Hostel,3.5,7,3,3,6.7,2,2,2,1,28.4,Low
3,Unspecified,B.Tech,3,CSE,Hostel,2.0,9,4,3,6.2,3,3,3,2,38.4,Moderate
4,IIIT Pune,B.Tech,4,ECE,Hostel,3.5,5,1,3,6.4,2,2,2,2,28.3,Low


### Data Quality Checks
We check column types, missing values (nulls), and duplicate rows to make sure the data is intact.


In [2]:
# Column information and missing values
print("--- Missing Values per Column ---")
print(df.isnull().sum())

# Duplicate rows check
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")


--- Missing Values per Column ---
institution                            0
degree                                 0
year                                   0
branch                                 0
living_arrangement                     0
study_hours_outside_classes_per_day    0
academic_tasks_per_week                0
academic_workload                      0
time_pressure                          0
avg_sleep_hours                        0
wakes_up_tired                         0
physical_activity_days_per_week        0
difficulty_concentrating               0
academic_emotional_exhaustion          0
burnout_risk_score                     0
burnout_risk_level                     0
dtype: int64

Duplicate rows: 0


### Inspecting and Cleaning the Institution Column
Some survey records have their institution listed as `Unspecified`. We remove these 100 rows so that our dataset only includes verified university students.


In [3]:
# Institution distribution in raw data
print("Institution counts before cleaning:")
print(df['institution'].value_counts())

# Filter out rows where institution is 'Unspecified' (ignoring case and whitespace)
unspecified_mask = df['institution'].astype(str).str.strip().str.lower() == 'unspecified'
df_clean = df[~unspecified_mask].copy().reset_index(drop=True)

print(f"\nRows before:  {len(df)}")
print(f"Rows removed: {unspecified_mask.sum()}")
print(f"Rows after:   {len(df_clean)}")


Institution counts before cleaning:
institution
IIIT Pune      1800
Unspecified     100
JNTUH           100
Name: count, dtype: int64

Rows before:  2000
Rows removed: 100
Rows after:   1900


### Comparing Target Distribution
We check `burnout_risk_level` before and after cleaning to ensure the class proportions remain balanced.


In [4]:
comparison = pd.DataFrame({
    'Before (Count)': df['burnout_risk_level'].value_counts(),
    'Before (%)': (df['burnout_risk_level'].value_counts(normalize=True) * 100).round(2),
    'After (Count)': df_clean['burnout_risk_level'].value_counts(),
    'After (%)': (df_clean['burnout_risk_level'].value_counts(normalize=True) * 100).round(2)
})
comparison


,Before (Count),Before (%),After (Count),After (%)
burnout_risk_level,,,,
Moderate,725,36.25,687,36.16
High,575,28.75,553,29.11
Low,481,24.05,455,23.95
Critical,219,10.95,205,10.79


### Saving the Cleaned Dataset
We save the cleaned dataset into `data/processed/`. The original raw dataset is preserved without changes.


In [5]:
processed_dir.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(processed_path, index=False)
print(f"Saved cleaned dataset to: {processed_path}")

# Quick reload check
df_check = pd.read_csv(processed_path)
print(f"Verified reloaded shape: {df_check.shape}")


Saved cleaned dataset to: ..\data\processed\cognivault_cleaned_dataset.csv
Verified reloaded shape: (1900, 16)
